# Sensitivity to Extreme Climate Events

## PCA on CropFusionNet Latent (Fusion-Layer) Representations

**Goal:** Extract the `pooled_output` vector (the `DynamicPyramidalPooling` output, shape `B × hidden_size`) for every sample,
classify each `(NUTS_ID, year)` sample as **negative extreme / normal / positive extreme** based on the detrended yield z-anomaly,
then apply PCA to check whether the model's internal representation separates these regime classes.

**What to look for:**

- Clear spatial separation → model encodes extreme climate conditions differently in latent space
- Overlapping clusters → model does not distinguish extremes well internally
- Direction of PC shift → which latent dimensions "activate" under stress


## Import libraries


In [ ]:
import os
import pickle
import sys
import warnings

source_folder = "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src"
sys.path.append(source_folder)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset.dataset import CropFusionNetDataset
from models.CropFusionNet.model import CropFusionNet
from utils.utils import load_config, set_seed

warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "DeJavu Serif"
plt.rcParams["font.serif"] = "Times New Roman"

set_seed(42)

out_dir = "/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/output/figures"

## Configuration


In [ ]:
CROPS = ["winter_wheat", "winter_barley", "silage_maize"]

CROP_LABELS = {
    "winter_wheat": "Winter Wheat",
    "winter_barley": "Winter Barley",
    "silage_maize": "Silage Maize",
}

CROP_MONTHS = {
    "winter_wheat": "Jul",
    "winter_barley": "Jul",
    "silage_maize": "Sep",
}

FORECAST_DIRS = {
    crop: f"/beegfs/halder/GITHUB/RESEARCH/crop-yield-forecasting-germany/src/train/forecast/{crop}/{month}"
    for crop, month in CROP_MONTHS.items()
}

# Extreme-year threshold (|z-anomaly_detrended| > Z_THRESH → extreme)
Z_THRESH = 1.5  # standard deviations; per-NUTS3 detrended yield

BATCH_SIZE = 128
N_PCA_COMPONENTS = 10  # number of PCs to compute

# Colour palette for year types
TYPE_PALETTE = {
    "Negative Extreme": "#fe4a49",  # red
    "Normal": "#fed766",  # yellow
    "Positive Extreme": "#6a994e",  # green
}

TYPE_MARKERS = {
    "Negative Extreme": "v",
    "Normal": "o",
    "Positive Extreme": "^",
}

## Extract Latent Representations from CropFusionNet

For each crop we:

1. Load the trained `best_model.pt` checkpoint.
2. Build a full dataset (train + val + test splits merged).
3. Run a no-grad forward pass and collect the `"latent"` key (`pooled_output`, shape `B × hidden_size`).
4. Store alongside `NUTS_ID`, `year`, and the de-scaled `target_yield`.


In [ ]:
def extract_latents_for_crop(crop_name: str) -> pd.DataFrame:
    """
    Load the CropFusionNet checkpoint for `crop_name`, run a full forward pass
    over train + val + test splits, and return a DataFrame with columns:
        NUTS_ID | year | target_yield | latent_0 … latent_{H-1}
    """
    global cfg, model_config, train_config
    cfg, model_config, train_config = load_config(crop_name)
    device = model_config["device"]
    month = CROP_MONTHS[crop_name]
    ckpt = os.path.join(FORECAST_DIRS[crop_name], "best_model.pt")

    # Load model
    model = CropFusionNet(model_config).to(device)
    state = torch.load(ckpt, map_location=device)
    model.load_state_dict(state)
    model.eval()
    print(f"  ✅ [{CROP_LABELS[crop_name]}] model loaded from {ckpt}")

    target_mean = cfg.scalers["yield_mean"]
    target_std = cfg.scalers["yield_std"]

    all_latents = []
    all_targets = []
    all_nuts_ids = []
    all_years = []

    # Iterate over all three splits
    for split in ("train", "val", "test"):
        dataset = CropFusionNetDataset(cfg, mode=split, scale=True)
        dataloader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
        )

        with torch.no_grad():
            for batch in tqdm(dataloader, desc=f"    {split}", leave=False):
                inputs = {
                    "inputs": batch["inputs"].to(device),
                    "identifier": batch["identifier"].to(device),
                    "mask": batch["mask"].to(device),
                    "variable_mask": batch["variable_mask"].to(device),
                }
                out = model(inputs)

                # latent: (B, hidden_size)  — pooled_output before projection
                latent_vec = out["latent"].detach().cpu().numpy()
                targets = batch["target"].numpy() * target_std + target_mean

                all_latents.append(latent_vec)
                all_targets.append(targets)
                all_nuts_ids.extend(batch["NUTS_ID"])
                all_years.extend([int(y) for y in batch["year"]])

    latent_matrix = np.concatenate(all_latents, axis=0)  # (N, hidden_size)
    targets_vec = np.concatenate(all_targets, axis=0)  # (N,)

    hidden_size = latent_matrix.shape[1]
    latent_cols = [f"latent_{i}" for i in range(hidden_size)]

    df = pd.DataFrame(latent_matrix, columns=latent_cols)
    df.insert(0, "NUTS_ID", all_nuts_ids)
    df.insert(1, "year", all_years)
    df.insert(2, "target_yield", targets_vec)

    print(f"  → {len(df):,} samples | latent dim = {hidden_size}")
    return df


# Run extraction for all crops
latent_data = {}
for crop in CROPS:
    print(f"\n🔍 Extracting latents: {CROP_LABELS[crop]}")
    latent_data[crop] = extract_latents_for_crop(crop)

print("\n✅ Latent extraction complete for all crops.")

## Label Extreme vs Normal Years

We compute a **per-NUTS3 detrended yield z-anomaly** directly from the `target_yield` column that was already collected during forward pass. The detrending uses a linear per-district trend (simple, reproducible). Samples are then labelled:

| Class                | Condition                         |
| -------------------- | --------------------------------- |
| **Negative Extreme** | z-anomaly < −Z_THRESH             |
| **Normal**           | −Z_THRESH ≤ z-anomaly ≤ +Z_THRESH |
| **Positive Extreme** | z-anomaly > +Z_THRESH             |


In [ ]:
from scipy.stats import linregress


def add_year_type_labels(df: pd.DataFrame, z_thresh: float = Z_THRESH) -> pd.DataFrame:
    """
    Add a `year_type` column to `df` (which must contain NUTS_ID, year, target_yield).

    Per-NUTS3 detrended residual is computed via ordinary linear regression on year.
    The per-NUTS3 z-anomaly of residuals classifies each sample.
    """
    df = df.copy()
    residuals = np.full(len(df), np.nan)

    for nuts_id, grp in df.groupby("NUTS_ID"):
        idx = grp.index
        years = grp["year"].values.astype(float)
        ylds = grp["target_yield"].values

        if len(grp) < 3:
            residuals[idx] = ylds - ylds.mean()
            continue

        slope, intercept, *_ = linregress(years, ylds)
        trend = intercept + slope * years
        residuals[idx] = ylds - trend

    df["residual"] = residuals

    # Per-NUTS3 z-score of the detrended residual
    df["resid_mean"] = df.groupby("NUTS_ID")["residual"].transform("mean")
    df["resid_std"] = (
        df.groupby("NUTS_ID")["residual"].transform("std").replace(0, np.nan)
    )
    df["z_anomaly"] = (df["residual"] - df["resid_mean"]) / df["resid_std"]

    def classify(z):
        if z < -z_thresh:
            return "Negative Extreme"
        elif z > z_thresh:
            return "Positive Extreme"
        else:
            return "Normal"

    df["year_type"] = df["z_anomaly"].apply(classify)
    df.drop(columns=["residual", "resid_mean", "resid_std"], inplace=True)

    return df


# Apply labelling to all crops
labelled_data = {}
for crop in CROPS:
    labelled_data[crop] = add_year_type_labels(latent_data[crop])
    vc = labelled_data[crop]["year_type"].value_counts()
    print(f"[{CROP_LABELS[crop]}]  {dict(vc)}")

print(f"\nZ-threshold used: ±{Z_THRESH}")

## PCA on Latent Representations

We fit a `StandardScaler + PCA` pipeline on the combined dataset (all samples from all year types). The first `N_PCA_COMPONENTS` principal components are retained.

**Visualisations produced:**

1. **Scree plot** — explained variance ratio per PC (helps determine how many PCs matter)
2. **2-D scatter (PC1 vs PC2)** — one panel per crop, coloured by `year_type`
3. **3-D scatter (PC1, PC2, PC3)** — interactive rotation reveals cluster depth


In [ ]:
def fit_pca(df: pd.DataFrame, n_components: int = N_PCA_COMPONENTS):
    """
    Fit StandardScaler + PCA on the latent columns of `df`.

    Returns
    -------
    pca_df   : DataFrame with PC1…PCn, NUTS_ID, year, target_yield, z_anomaly, year_type
    pca      : fitted PCA object
    scaler   : fitted StandardScaler
    lat_cols : list of latent column names used
    """
    lat_cols = [c for c in df.columns if c.startswith("latent_")]
    X = df[lat_cols].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    n_comp = min(n_components, X_scaled.shape[1], X_scaled.shape[0])
    pca = PCA(n_components=n_comp, random_state=42)
    X_pca = pca.fit_transform(X_scaled)

    pc_cols = [f"PC{i+1}" for i in range(n_comp)]
    pca_df = pd.DataFrame(X_pca, columns=pc_cols, index=df.index)
    meta_cols = ["NUTS_ID", "year", "target_yield", "z_anomaly", "year_type"]
    for col in meta_cols:
        if col in df.columns:
            pca_df[col] = df[col].values

    return pca_df, pca, scaler, lat_cols


# Fit PCA for each crop
pca_results = {}
for crop in CROPS:
    df_lab = labelled_data[crop]
    pca_df, pca_obj, scaler_obj, lat_cols = fit_pca(
        df_lab, n_components=N_PCA_COMPONENTS
    )
    pca_results[crop] = {
        "pca_df": pca_df,
        "pca": pca_obj,
        "scaler": scaler_obj,
        "lat_cols": lat_cols,
    }
    evr = pca_obj.explained_variance_ratio_
    cumvar = np.cumsum(evr)
    print(
        f"[{CROP_LABELS[crop]}]  PC1={evr[0]:.1%}  PC2={evr[1]:.1%}  "
        f"PC3={evr[2]:.1%}  cumvar@{len(evr)}={cumvar[-1]:.1%}"
    )

print("\n✅ PCA fitted for all crops.")

### Scree Plot


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(
    1, len(CROPS), figsize=(4.5 * len(CROPS), 3.5), dpi=150, sharey=False
)
panel_labels_scree = ["(a)", "(b)", "(c)"]

for i, (ax, crop, panel) in enumerate(zip(axes, CROPS, panel_labels_scree)):
    pca_obj = pca_results[crop]["pca"]
    evr = pca_obj.explained_variance_ratio_
    cumvar = np.cumsum(evr)
    pcs = np.arange(1, len(evr) + 1)

    bars = ax.bar(
        pcs,
        evr * 100,
        color="#4393c3",
        alpha=0.8,
        label="Individual",
        edgecolor="k",
        linewidth=0.8,
    )
    ax.bar_label(bars, fmt="%.1f%%", fontsize=8, padding=2)
    ax.set_ylim(0, 65)
    ax.set_xlabel("Principal components", fontsize=12)

    ax2 = ax.twinx()
    ax2.plot(
        pcs,
        cumvar * 100,
        "o-",
        color="#d73027",
        linewidth=1.5,
        markersize=4,
        label="Cumulative",
    )
    ax2.axhline(80, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)

    ax2.set_ylim(0, 105)
    ax2.tick_params(labelsize=12)

    ax.set_title(CROP_LABELS[crop], fontsize=12, fontweight="bold")
    ax.set_xticks(pcs)
    ax.tick_params(labelsize=10)

    ax.text(
        0.03,
        0.97,
        panel,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
    )

    if i == 0:
        ax.set_ylabel("Explained variance (%)", fontsize=12)
    else:
        ax.set_ylabel(None)

    if i == len(CROPS) - 1:
        ax2.set_ylabel("Cumulative variance (%)", fontsize=12)
    else:
        ax2.set_ylabel(None)

    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()

fig.legend(
    lines1 + lines2,
    labs1 + labs2,
    loc="lower center",
    ncol=2,
    bbox_to_anchor=(0.5, -0.1),
    fontsize=12,
    frameon=False,
)

plt.tight_layout()
fig.subplots_adjust(bottom=0.15)

save_path = os.path.join(out_dir, "final", "pca_latent_scree.png")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.savefig(save_path, bbox_inches="tight", dpi=300)
print(f"Saved → {save_path}")
plt.show()

### 2-D PCA Scatter (PC1 vs PC2) coloured by Year Type


In [ ]:
TYPE_ORDER = ["Negative Extreme", "Normal", "Positive Extreme"]
ALPHA_MAP = {"Negative Extreme": 0.75, "Normal": 0.25, "Positive Extreme": 0.75}
SIZE_MAP = {"Negative Extreme": 18, "Normal": 8, "Positive Extreme": 18}

fig, axes = plt.subplots(1, len(CROPS), figsize=(5.5 * len(CROPS), 5), dpi=150)
panel_labels_2d = ["(a)", "(b)", "(c)"]

for ax, crop, panel in zip(axes, CROPS, panel_labels_2d):
    pca_df = pca_results[crop]["pca_df"]
    pca_obj = pca_results[crop]["pca"]
    evr = pca_obj.explained_variance_ratio_

    # Plot Normal first (background), then extremes on top
    for ytype in TYPE_ORDER:
        sub = pca_df[pca_df["year_type"] == ytype]
        ax.scatter(
            sub["PC1"],
            sub["PC2"],
            c=TYPE_PALETTE[ytype],
            marker=TYPE_MARKERS[ytype],
            s=SIZE_MAP[ytype],
            alpha=ALPHA_MAP[ytype],
            linewidths=0,
            label=ytype,
            zorder=3 if ytype != "Normal" else 1,
        )

    # 95% confidence ellipses (Gaussian approx.) for extreme classes
    from matplotlib.patches import Ellipse
    import matplotlib.transforms as mtransforms

    def draw_confidence_ellipse(ax, x, y, color, n_std=2.0, **kwargs):
        if len(x) < 3:
            return
        cov = np.cov(x, y)
        mean = [np.mean(x), np.mean(y)]
        vals, vecs = np.linalg.eigh(cov)
        order = vals.argsort()[::-1]
        vals, vecs = vals[order], vecs[:, order]
        angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        width = 2 * n_std * np.sqrt(vals[0])
        height = 2 * n_std * np.sqrt(vals[1])
        ellipse = Ellipse(
            xy=mean,
            width=width,
            height=height,
            angle=angle,
            edgecolor=color,
            facecolor="none",
            linewidth=1.5,
            linestyle="--",
            zorder=5,
            **kwargs,
        )
        ax.add_patch(ellipse)

    for ytype in ["Negative Extreme", "Positive Extreme"]:
        sub = pca_df[pca_df["year_type"] == ytype]
        draw_confidence_ellipse(
            ax, sub["PC1"].values, sub["PC2"].values, color=TYPE_PALETTE[ytype]
        )

    # centroid markers
    for ytype in TYPE_ORDER:
        sub = pca_df[pca_df["year_type"] == ytype]
        cx, cy = sub["PC1"].mean(), sub["PC2"].mean()
        ax.plot(
            cx,
            cy,
            marker="*",
            markersize=18,
            color=TYPE_PALETTE[ytype],
            markeredgecolor="k",
            markeredgewidth=1,
            zorder=6,
        )

    ax.set_title(CROP_LABELS[crop], fontsize=14, fontweight="bold")
    ax.set_xlabel(f"PC1  ({evr[0]:.1%} var.)", fontsize=14)
    ax.set_ylabel(f"PC2  ({evr[1]:.1%} var.)", fontsize=14)
    ax.tick_params(labelsize=14)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
    ax.text(
        0.03,
        0.97,
        panel,
        transform=ax.transAxes,
        fontsize=18,
        fontweight="bold",
        va="top",
    )

from matplotlib.lines import Line2D

handles = [
    Line2D(
        [0],
        [0],
        marker=TYPE_MARKERS[t],
        color="w",
        markerfacecolor=TYPE_PALETTE[t],
        markersize=14,
        label=t,
    )
    for t in TYPE_ORDER
]
handles.append(
    Line2D(
        [0],
        [0],
        marker="*",
        color="w",
        markerfacecolor="grey",
        markeredgecolor="k",
        markersize=14,
        label="Centroid",
    )
)

fig.legend(
    handles=handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0),
    ncol=4,
    fontsize=14,
    title="Year type",
    title_fontsize=14,
    frameon=False,
)

fig.subplots_adjust(bottom=0.2, wspace=0.25)

plt.tight_layout()

save_path = os.path.join(out_dir, "final", "pca_latent_2d_scatter.png")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.savefig(save_path, bbox_inches="tight", dpi=150)
print(f"Saved → {save_path}")
plt.show()

### 3-D PCA Scatter (PC1, PC2, PC3)


In [ ]:
fig = plt.figure(figsize=(6.5 * len(CROPS), 6), dpi=130)

for idx, crop in enumerate(CROPS):
    pca_df = pca_results[crop]["pca_df"]
    pca_obj = pca_results[crop]["pca"]
    evr = pca_obj.explained_variance_ratio_

    ax = fig.add_subplot(1, len(CROPS), idx + 1, projection="3d")

    for ytype in TYPE_ORDER:
        sub = pca_df[pca_df["year_type"] == ytype]
        ax.scatter(
            sub["PC1"],
            sub["PC2"],
            sub["PC3"],
            c=TYPE_PALETTE[ytype],
            marker=TYPE_MARKERS[ytype],
            s=SIZE_MAP[ytype],
            alpha=ALPHA_MAP[ytype],
            linewidths=0,
            label=ytype,
            depthshade=True,
        )

    # centroid markers
    for ytype in TYPE_ORDER:
        sub = pca_df[pca_df["year_type"] == ytype]
        ax.scatter(
            sub["PC1"].mean(),
            sub["PC2"].mean(),
            sub["PC3"].mean(),
            c=TYPE_PALETTE[ytype],
            marker="*",
            s=200,
            edgecolors="k",
            linewidths=0.7,
            zorder=10,
        )

    ax.set_title(CROP_LABELS[crop], fontsize=10, fontweight="bold", pad=6)
    ax.set_xlabel(f"PC1 ({evr[0]:.1%})", fontsize=7, labelpad=4)
    ax.set_ylabel(f"PC2 ({evr[1]:.1%})", fontsize=7, labelpad=4)
    ax.set_zlabel(f"PC3 ({evr[2]:.1%})", fontsize=7, labelpad=4)
    ax.tick_params(labelsize=6)
    ax.view_init(elev=20, azim=135)

    ax.text2D(
        0.03,
        0.97,
        f"({chr(97+idx)})",
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="top",
    )

    if idx == len(CROPS) - 1:
        handles = [
            plt.scatter(
                [], [], c=TYPE_PALETTE[t], marker=TYPE_MARKERS[t], s=30, label=t
            )
            for t in TYPE_ORDER
        ]
        ax.legend(
            handles=handles,
            fontsize=7,
            loc="upper left",
            title="Year type",
            title_fontsize=7,
            framealpha=0.8,
        )

plt.suptitle(
    "3-D PCA — CropFusionNet Fusion-Layer Representations",
    fontsize=12,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()

save_path = os.path.join(out_dir, "final", "pca_latent_3d_scatter.png")
fig.savefig(save_path, bbox_inches="tight", dpi=300)
print(f"Saved → {save_path}")
plt.show()

## PC Centroid Shift Analysis

**Question:** How far do the mean representations of _negative extreme_, _normal_, and _positive extreme_ samples lie apart in PCA space?

We compute the **Euclidean distance** between the centroids of each pair in the PC1–PC3 subspace, and also plot a **centroid shift bar chart** showing the displacement of extremes from the normal centroid along each PC axis.


In [ ]:
N_SHIFT_PCS = 5  # how many PCs to show in the shift bar chart

print("=" * 60)
print("  Centroid Euclidean Distances in PC1–PC3 space")
print("=" * 60)

shift_records = []

for crop in CROPS:
    pca_df = pca_results[crop]["pca_df"]
    pc_cols = ["PC1", "PC2", "PC3"]

    centroids = {}
    for ytype in TYPE_ORDER:
        sub = pca_df[pca_df["year_type"] == ytype][pc_cols]
        centroids[ytype] = sub.mean().values

    # Euclidean distances
    neg_c = centroids["Negative Extreme"]
    pos_c = centroids["Positive Extreme"]
    norm_c = centroids["Normal"]

    d_neg_norm = np.linalg.norm(neg_c - norm_c)
    d_pos_norm = np.linalg.norm(pos_c - norm_c)
    d_neg_pos = np.linalg.norm(neg_c - pos_c)

    print(f"\n[{CROP_LABELS[crop]}]")
    print(f"  Neg extreme ↔ Normal :  {d_neg_norm:.4f}")
    print(f"  Pos extreme ↔ Normal :  {d_pos_norm:.4f}")
    print(f"  Neg extreme ↔ Pos extreme: {d_neg_pos:.4f}")

    # Per-PC shift (signed) for each extreme vs normal centroid
    shift_pcs = [f"PC{i+1}" for i in range(N_SHIFT_PCS)]
    for ytype in ["Negative Extreme", "Positive Extreme"]:
        sub = pca_df[pca_df["year_type"] == ytype][shift_pcs]
        norm_sub = pca_df[pca_df["year_type"] == "Normal"][shift_pcs]
        delta = sub.mean().values - norm_sub.mean().values
        for pc_idx, pc_name in enumerate(shift_pcs):
            shift_records.append(
                {
                    "crop": CROP_LABELS[crop],
                    "year_type": ytype,
                    "PC": pc_name,
                    "shift": delta[pc_idx],
                }
            )

print("\n✅ Distances computed.")
shift_df = pd.DataFrame(shift_records)
shift_df.head(10)

### Centroid Shift Bar Chart (extreme vs. normal, per PC)


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(
    1, len(CROPS), figsize=(5 * len(CROPS), 4), dpi=150, sharey=False
)
panel_labels_shift = ["(a)", "(b)", "(c)"]
bar_width = 0.32

for i, (ax, crop, panel) in enumerate(zip(axes, CROPS, panel_labels_shift)):
    crop_label = CROP_LABELS[crop]
    sub = shift_df[shift_df["crop"] == crop_label]
    pcs = sorted(sub["PC"].unique())
    x = np.arange(len(pcs))

    for j, ytype in enumerate(["Negative Extreme", "Positive Extreme"]):
        vals = sub[sub["year_type"] == ytype].set_index("PC").loc[pcs, "shift"].values
        offset = (j - 0.5) * bar_width
        bars = ax.bar(
            x + offset,
            vals,
            width=bar_width,
            color=TYPE_PALETTE[ytype],
            alpha=0.8,
            label=ytype,
            edgecolor="k",
            linewidth=0.4,
        )
        ax.bar_label(bars, fmt="%+.2f", padding=4, rotation=90, fontsize=12)

    max_abs_val = sub["shift"].abs().max()
    ax.set_ylim(-max_abs_val * 2, max_abs_val * 2)

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(crop_label, fontsize=14, fontweight="bold")
    ax.set_xlabel("Principal Component", fontsize=14)

    if i == 0:
        ax.set_ylabel("Centroid shift vs. Normal", fontsize=14)
    else:
        ax.set_ylabel(None)

    ax.set_xticks(x)
    ax.set_xticklabels(pcs, fontsize=14)
    ax.tick_params(axis="y", labelsize=14)
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.5)
    ax.text(
        0.90,
        0.97,
        panel,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        va="top",
    )

    if i == len(CROPS) - 1:
        handles, labels = ax.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.07),
    ncol=2,
    fontsize=14,
    frameon=False,
)

plt.tight_layout()

fig.subplots_adjust(bottom=0.2)

save_path = os.path.join(out_dir, "final", "pca_latent_centroid_shift.png")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.savefig(save_path, bbox_inches="tight", dpi=300)
print(f"Saved → {save_path}")
plt.show()

## Year-Level Mean PC Trajectory

We compute the **year-level mean PC1 and PC2** across all NUTS3 districts to see how the model's aggregate latent representation changes from year to year. Drought years should stand out as deviations from the mean trajectory.


In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from adjustText import adjust_text

DROUGHT_YEARS = [2003, 2018, 2022]

fig, axes = plt.subplots(1, len(CROPS), figsize=(5 * len(CROPS), 5), dpi=150)
panel_labels_traj = ["(a)", "(b)", "(c)"]

for ax, crop, panel in zip(axes, CROPS, panel_labels_traj):
    pca_df = pca_results[crop]["pca_df"]
    pca_obj = pca_results[crop]["pca"]
    evr = pca_obj.explained_variance_ratio_

    # Year-mean centroids
    yearly = (
        pca_df.groupby("year")[["PC1", "PC2", "z_anomaly"]]
        .mean()
        .reset_index()
        .sort_values("year")
    )

    # Colour by z-anomaly magnitude
    z_clip = yearly["z_anomaly"].clip(-3, 3)
    vmax = max(abs(z_clip.min()), abs(z_clip.max()), 0.1)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    sc = ax.scatter(
        yearly["PC1"],
        yearly["PC2"],
        c=z_clip,
        cmap="RdYlGn",
        norm=norm,
        s=60,
        zorder=3,
        linewidths=0.5,
        edgecolors="k",
    )

    ax.margins(0.15)

    # 1. Create an empty list to store the text objects
    texts = []

    # Annotate each year
    for _, row in yearly.iterrows():
        yr = int(row["year"])
        fw = "bold" if yr in DROUGHT_YEARS else "normal"
        col = "#d73027" if yr in DROUGHT_YEARS else "black"

        # 2. Use ax.text and append the result to our texts list
        t = ax.text(
            row["PC1"],
            row["PC2"],
            str(yr),
            fontsize=10,
            color=col,
            fontweight=fw,
        )
        texts.append(t)

    # 3. Call adjust_text to magically fix all overlaps and boundary issues
    adjust_text(
        texts,
        ax=ax,
        x=yearly["PC1"].values,
        y=yearly["PC2"].values,
        arrowprops=dict(arrowstyle="-", color="gray", lw=0.5, alpha=0.5),
    )

    cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.18, orientation="horizontal")
    cbar.set_label("Mean z-anomaly", fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    ax.set_title(CROP_LABELS[crop], fontsize=14, fontweight="bold")
    ax.set_xlabel(f"PC1  ({evr[0]:.1%} var.)", fontsize=14)
    ax.set_ylabel(f"PC2  ({evr[1]:.1%} var.)", fontsize=14)
    ax.tick_params(labelsize=14)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.4)
    ax.text(
        0.88,
        0.03,
        panel,
        transform=ax.transAxes,
        fontsize=18,
        fontweight="bold",
        va="bottom",
    )

plt.tight_layout()

save_path = os.path.join(out_dir, "final", "pca_latent_year_trajectory.png")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.savefig(save_path, bbox_inches="tight", dpi=300)
print(f"Saved → {save_path}")
plt.show()